# Notebook 9: Post-hoc Linear Probe Baseline

A post-hoc analysis added after the project was frozen. It answers one question: how much do the LoRA adapters
add over a plain linear classifier on frozen DINOv3 features?

For each target it fits a logistic-regression probe on the frozen backbone's pooled features of the existing
train split, then scores the probe and the released adapter on the same in-distribution test split and the same
held-out OOD split:

- accuracy, balanced accuracy and macro F1;
- OOD AUROC for the probe's MSP, max logit, energy and kNN scores, and for the adapter's calibrated primary score;
- AURC and AUGRC on the ID test split, with softmax confidence as the acceptance score.

Nothing here changes a released adapter, dataset or README number. Results are written to `OUTPUT_DIR`
with per-sample rows, so the curves can be redrawn without a GPU.

Needs a GPU runtime, `HF_TOKEN` with access to the gated DINOv3 weights, and `AADS_GITHUB_RELEASE_READ_TOKEN`
for the private dataset release.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

CLONE_TARGET = Path('/content/bitirmeprojesi')
REPO_URL = os.environ.get('AADS_REPO_URL', 'https://github.com/EfeErim/aads-open-world-plant-disease.git')
if not (CLONE_TARGET / 'src').is_dir() and not (Path.cwd() / 'src').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CLONE_TARGET)], check=True)
for candidate in (Path.cwd(), CLONE_TARGET):
    if (candidate / 'src').is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from src.colab.notebooks_bootstrap import run_notebook_bootstrap
from src.colab.repo_bootstrap import login_and_check_hf_token

BOOTSTRAP = run_notebook_bootstrap('Notebook 9: Post-hoc Linear Probe Baseline')
ROOT = Path(BOOTSTRAP['ROOT'])
login_and_check_hf_token()

In [ ]:
TARGETS = [
    'apricot__fruit', 'apricot__leaf', 'grape__fruit', 'grape__leaf',
    'strawberry__fruit', 'strawberry__leaf', 'tomato__fruit', 'tomato__leaf',
]
DATASET_RELEASE_REPOSITORY = 'EfeErim/aads-open-world-plant-disease'
DATASET_RELEASE_TAG = 'aads-dataset-v1.0.0'
DATASET_RELEASE_CACHE_ROOT = '.runtime_tmp/dataset_release_cache'
SCORE_RELEASED_ADAPTERS = True
OUTPUT_DIR = '.runtime_tmp/posthoc_linear_probe'
DEVICE = 'cuda'
BATCH_SIZE = 32
CONFIG_ENV = 'colab'

In [ ]:
import os

from src.data.dataset_release_runtime import fetch_materialize_dataset_release

if not os.environ.get('AADS_GITHUB_RELEASE_READ_TOKEN'):
    try:
        from google.colab import userdata

        os.environ['AADS_GITHUB_RELEASE_READ_TOKEN'] = str(userdata.get('AADS_GITHUB_RELEASE_READ_TOKEN') or '').strip()
    except Exception:  # Colab secret access raises provider-specific exceptions.
        pass

DATASET_REPORT = fetch_materialize_dataset_release(
    root=ROOT,
    repository=DATASET_RELEASE_REPOSITORY,
    release_tag=DATASET_RELEASE_TAG,
    targets=TARGETS,
    cache_root=DATASET_RELEASE_CACHE_ROOT,
)
print('[DATASET]', DATASET_REPORT['release_tag'], 'verified =', DATASET_REPORT['verified'])

ADAPTER_ROOT = None
if SCORE_RELEASED_ADAPTERS:
    from scripts.notebook_helpers.cell_script_runner import run_cell_script

    run_cell_script('nb8_cell04_adapter_release_fetch.py', globals())

In [ ]:
import gc

import torch

from scripts.posthoc_linear_probe import run as run_posthoc_probe

POSTHOC_SUMMARIES = {}
for target in TARGETS:
    target_root = Path(DATASET_REPORT['selected_dataset_roots'][target])
    crop, part = target.split('__')
    adapter_dir = Path(ADAPTER_ROOT) / crop / part / 'continual_sd_lora_adapter' if ADAPTER_ROOT else None
    print(f'[PROBE] {target}')
    POSTHOC_SUMMARIES[target] = run_posthoc_probe(
        dataset_root=target_root.parent,
        target=target,
        output_dir=ROOT / OUTPUT_DIR,
        adapter_dir=adapter_dir,
        device=DEVICE,
        batch_size=BATCH_SIZE,
    )
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
import pandas as pd

rows = []
for target, summary in POSTHOC_SUMMARIES.items():
    for model_name in ('probe', 'adapter'):
        result = summary.get(model_name)
        if not result:
            continue
        row = {'target': target, 'model': model_name, **summary['sample_counts']}
        row.update(result['classification'])
        row.update({f'auroc_{name}': value for name, value in result['ood_auroc'].items()})
        row.update(result['selective'])
        rows.append(row)

POSTHOC_TABLE = pd.DataFrame(rows).round(4)
POSTHOC_TABLE.to_csv(ROOT / OUTPUT_DIR / 'posthoc_comparison.csv', index=False)
POSTHOC_TABLE